# Piper TTS Voice Training for Speech2Text App

This notebook trains a **Piper TTS voice model** using recordings from your Speech2Text Android app.

## What is Piper?
Piper is a fast, local neural text-to-speech system that:
- ✅ Runs completely offline
- ✅ Works on Android devices
- ✅ Produces high-quality speech
- ✅ Supports multiple languages (German, English, Polish, etc.)

## Prerequisites:
- Exported training data from Speech2Text app to Google Drive
- At least 30-60 minutes of clear audio recordings
- GPU runtime enabled in Colab (Runtime → Change runtime type → GPU)

## Training Time:
- With checkpoint (recommended): 2-4 hours
- From scratch: 8-12+ hours

## Steps:
1. Setup GPU and system packages
2. Clone and install Piper training tools
3. Mount Google Drive and prepare data
4. Configure training parameters
5. Start training
6. Export trained model
7. Download for use in your app

## Step 1: Check GPU and System Info

In [ ]:
# Check GPU availability
import torch, platform, sys

print("📊 System Information:")
print(f"   Python: {sys.version.split()[0]}")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"\n✅ GPU Details:")
    !nvidia-smi
else:
    print("\n⚠️ No GPU detected!")
    print("   Please enable GPU: Runtime → Change runtime type → GPU → Save")
    print("   Then restart this notebook.")

## Step 2: Install System Packages

In [ ]:
# Install required system packages including eSpeak
print("📦 Installing system packages...\n")

!sudo apt-get update -y
!sudo apt-get install -y build-essential cmake ninja-build espeak-ng espeak-ng-data libespeak-ng-dev pkg-config ffmpeg

# Verify eSpeak installation
print("\n✅ Checking eSpeak-NG version:")
!pkg-config --modversion espeak-ng

## Step 3: Clone Piper Training Repository

In [ ]:
# Clone the Piper GPL training repository
import os

os.chdir('/content')

# Remove if exists
if os.path.exists('piper1-gpl'):
    !rm -rf piper1-gpl

print("📥 Cloning Piper training repository...\n")
!git clone https://github.com/OHF-voice/piper1-gpl.git

os.chdir('piper1-gpl')
print(f"\n✅ Repository cloned to: {os.getcwd()}")

## Step 4: Install Python Dependencies

In [ ]:
# Install Piper in editable mode with training dependencies
print("📦 Installing Python dependencies...\n")
print("   This may take 3-5 minutes...\n")

!python3 -m pip install --upgrade pip setuptools wheel
!python3 -m pip install -e ".[train]"

print("\n✅ Python dependencies installed!")

## Step 5: Build Alignment Extension

In [ ]:
# Build the Cython extension for monotonic alignment
os.chdir('/content/piper1-gpl')

print("🔨 Building alignment extension...\n")

!chmod +x ./build_monotonic_align.sh
!./build_monotonic_align.sh

print("\n✅ Alignment extension built!")

## Step 6: Development Build

In [ ]:
# Install additional build tools
!python3 -m pip install --upgrade pip setuptools wheel scikit-build cmake ninja

In [ ]:
# Build extensions in place
os.chdir('/content/piper1-gpl')

print("🔨 Building Piper extensions...\n")
!python3 setup.py build_ext --inplace -v

print("\n✅ Build complete!")

## Step 7: Mount Google Drive

In [ ]:
from google.colab import drive
import os

print("📂 Mounting Google Drive...\n")
drive.mount('/content/drive')

# Verify training data location
TRAINING_DATA_PATH = '/content/drive/MyDrive/TTS_Voice_Samples'

if os.path.exists(TRAINING_DATA_PATH):
    wav_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.wav')]
    csv_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.csv')]
    
    print(f"\n✅ Found training data:")
    print(f"   WAV files: {len(wav_files)}")
    print(f"   CSV files: {len(csv_files)}")
else:
    print(f"\n❌ Training data not found at: {TRAINING_DATA_PATH}")
    print(f"   Please export data from your Speech2Text app first!")

## Step 8: Prepare Training Data

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# Setup paths
DATA_ROOT = Path("/content/drive/MyDrive/piper_training")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

AUDIO_DIR = DATA_ROOT / "wavs"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

METADATA_CSV = DATA_ROOT / "metadata.csv"

print("📦 Preparing training data...\n")

# Read CSV from app export
csv_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.csv')]

if csv_files:
    app_csv = os.path.join(TRAINING_DATA_PATH, csv_files[0])
    
    # Read app CSV: filename|text|language
    with open(app_csv, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Convert to Piper format: filename|text (no language column)
    piper_lines = []
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            filename = parts[0]
            text = parts[1]
            
            # Copy WAV file
            src = Path(TRAINING_DATA_PATH) / filename
            dst = AUDIO_DIR / filename
            
            if src.exists():
                shutil.copy2(src, dst)
                piper_lines.append(f"{filename}|{text}")
    
    # Write Piper metadata.csv
    with open(METADATA_CSV, 'w', encoding='utf-8') as f:
        f.write('\n'.join(piper_lines))
    
    print(f"✅ Prepared {len(piper_lines)} training samples")
    print(f"   Audio files: {AUDIO_DIR}")
    print(f"   Metadata: {METADATA_CSV}")
    
    # Show sample
    if piper_lines:
        print(f"\n📝 Sample entry:")
        print(f"   {piper_lines[0][:80]}...")
else:
    print("❌ No CSV file found in training data!")

## Step 9: Configure Training Parameters

In [ ]:
from pathlib import Path

# ==== AUTO-DETECT LANGUAGE FROM RECORDINGS ====

# Read language from original CSV export
csv_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.csv')]
detected_language = None

if csv_files:
    app_csv = os.path.join(TRAINING_DATA_PATH, csv_files[0])
    with open(app_csv, 'r', encoding='utf-8') as f:
        first_line = f.readline().strip()
        parts = first_line.split('|')
        if len(parts) >= 3:
            detected_language = parts[2].strip()

# Map app language codes to eSpeak voice codes
language_map = {
    '🇩🇪 Deutsch': 'de',
    '🇬🇧 English': 'en-us',
    '🇵🇱 Polski': 'pl'
}

# Checkpoint URLs for different languages
checkpoint_map = {
    'de': 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/de/de_DE/thorsten/medium/epoch%3D2164-step%3D1355540.ckpt',
    'en-us': 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt',
    'pl': ''  # No medium checkpoint available for Polish
}

# ==== TRAINING CONFIGURATION ====

# Voice settings
VOICE_NAME = "my_speech2text_voice"

# Auto-detect language or fallback to default
ESPEAK_VOICE = language_map.get(detected_language, 'de')
CKPT_URL = checkpoint_map.get(ESPEAK_VOICE, '')

# Audio settings
SAMPLE_RATE_HZ = 22050

# Training settings
BATCH_SIZE = 8  # Reduce to 4 if you get out of memory errors

# Paths
CACHE_DIR = Path("/content/piper_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = DATA_ROOT / f"{VOICE_NAME}.json"

print("⚙️ Training Configuration:")
print(f"   Voice name: {VOICE_NAME}")
if detected_language:
    print(f"   Detected from recordings: {detected_language}")
print(f"   Language (eSpeak): {ESPEAK_VOICE}")
print(f"   Sample rate: {SAMPLE_RATE_HZ} Hz")
print(f"   Batch size: {BATCH_SIZE}")
if CKPT_URL:
    print(f"   Using checkpoint: Yes (faster training)")
else:
    print(f"   Using checkpoint: No (training from scratch, slower)")
    print(f"   ⚠️ Note: Polish has no pre-trained checkpoint, training will take longer")
print(f"\n📁 Paths:")
print(f"   CSV: {METADATA_CSV.exists()} - {METADATA_CSV}")
print(f"   Audio dir: {AUDIO_DIR.exists()} - {AUDIO_DIR}")
print(f"   Cache: {CACHE_DIR}")
print(f"   Config: {CONFIG_PATH}")

## Step 10: Verify eSpeak Voice

In [ ]:
# Check available eSpeak voices
print("🗣️ Available eSpeak voices:\n")
!espeak-ng --voices | grep -E "de|en|pl" | head -n 20

print(f"\n✅ Selected voice: {ESPEAK_VOICE}")
print(f"   Make sure this matches your recording language!")

## Step 11: Verify Training Data

In [ ]:
import pandas as pd

# Read and verify metadata
if METADATA_CSV.exists():
    df = pd.read_csv(str(METADATA_CSV), sep="|", header=None, names=["audio", "text"])
    
    print(f"📊 Training Data Summary:")
    print(f"   Total samples: {len(df)}")
    print(f"\n📝 First 5 entries:")
    print(df.head())
    
    # Check if audio files exist
    print(f"\n🔍 Checking audio files...")
    missing = [a for a in df["audio"].head(5) if not (AUDIO_DIR / str(a)).exists()]
    
    if missing:
        print(f"   ⚠️ Missing files: {missing}")
    else:
        print(f"   ✅ All checked files exist!")
else:
    print(f"❌ Metadata CSV not found at: {METADATA_CSV}")

## Step 12: Start Training! 🚀

**This will take 2-4 hours with a checkpoint, or 8-12+ hours without.**

You can close this tab and come back later - the training will continue in the background.

In [ ]:
# Start Piper training
print("🚀 Starting Piper training...\n")
print("   ⏱️ This will take several hours.")
print("   💡 You can close this tab - training continues in background.")
print("   📊 Monitor progress below...\n")

!python3 -m piper.train fit \
  --data.voice_name "{VOICE_NAME}" \
  --data.csv_path "{str(METADATA_CSV)}" \
  --data.audio_dir "{str(AUDIO_DIR)}" \
  --model.sample_rate {SAMPLE_RATE_HZ} \
  --data.espeak_voice "{ESPEAK_VOICE}" \
  --data.cache_dir "{str(CACHE_DIR)}" \
  --data.config_path "{str(CONFIG_PATH)}" \
  --data.batch_size {BATCH_SIZE} \
  --ckpt_path "{CKPT_URL}"

print("\n\n✅ Training complete!")

## Step 13: Export to ONNX Format

In [ ]:
# Find the latest checkpoint
import glob

checkpoint_pattern = "/content/piper1-gpl/lightning_logs/version_*/checkpoints/*.ckpt"
checkpoints = sorted(glob.glob(checkpoint_pattern))

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f"📦 Found checkpoint: {latest_checkpoint}")
    
    output_onnx = "/content/drive/MyDrive/piper_training/model.onnx"
    
    print(f"\n🔄 Exporting to ONNX format...")
    !python3 -m piper.train.export_onnx \
      --checkpoint "{latest_checkpoint}" \
      --output-file "{output_onnx}"
    
    print(f"\n✅ Model exported to: {output_onnx}")
else:
    print("❌ No checkpoint found. Make sure training completed successfully.")

## Step 14: Copy Config File

In [ ]:
# Copy the config JSON file
import shutil

if CONFIG_PATH.exists():
    config_dest = "/content/drive/MyDrive/piper_training/model.onnx.json"
    shutil.copy2(str(CONFIG_PATH), config_dest)
    
    print(f"✅ Config copied to: {config_dest}")
    print(f"\n📦 Your trained model files:")
    print(f"   1. model.onnx")
    print(f"   2. model.onnx.json")
    print(f"\n   Both files are in: /content/drive/MyDrive/piper_training/")
else:
    print(f"❌ Config file not found at: {CONFIG_PATH}")

## Step 15: Test Your Voice Model!

In [ ]:
from IPython.display import Audio, display
import subprocess

# Test text based on detected language
test_texts = {
    "de": "Hallo! Dies ist meine trainierte Stimme mit Piper Text to Speech.",
    "en-us": "Hello! This is my trained voice using Piper text to speech.",
    "pl": "Cześć! To jest mój wyszkolony głos używający Piper text to speech."
}

test_text = test_texts.get(ESPEAK_VOICE, "Hello, this is a test.")

print(f"🎤 Testing your voice model...")
print(f"   Language: {ESPEAK_VOICE}")
print(f"   Text: {test_text}")

output_audio = "/content/test_output.wav"
model_path = "/content/drive/MyDrive/piper_training/model.onnx"

# Generate speech
!echo "{test_text}" | /content/piper1-gpl/piper \
  --model "{model_path}" \
  --output_file "{output_audio}"

print(f"\n🔊 Generated audio:")
display(Audio(output_audio))

print(f"\n💡 If the quality is not good enough:")
print(f"   - Record more training data (60+ minutes recommended)")
print(f"   - Ensure audio quality is consistent")
print(f"   - Try training for more epochs")

## ✅ Training Complete!

### Your trained model files:
- `model.onnx` - The trained neural network
- `model.onnx.json` - Configuration file

### Location:
`/content/drive/MyDrive/piper_training/`

### Next Steps:
1. Download both files from Google Drive
2. Use them with Piper TTS in your Android app
3. Or use with desktop Piper: https://github.com/rhasspy/piper

### Integration with Android:
```kotlin
// Example usage in your Speech2Text app
val piperTts = PiperTTS(
    modelPath = "path/to/model.onnx",
    configPath = "path/to/model.onnx.json"
)

val audioData = piperTts.synthesize("Your text here")
```

### Congratulations! 🎉
You've successfully trained a custom voice model with Piper TTS!